In [1]:
# ==============================================================================
# 1. Librerías estándar del sistema
# ==============================================================================
import datetime
import os
import sys
import warnings

import os
import sys

# 1. Obtener la ruta del directorio raíz del proyecto (un nivel arriba de 'notebooks/')
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# 2. Insertar la ruta al inicio de sys.path
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 3. Ahora las importaciones de src funcionarán correctamente
from src.features.build_features import build_features
from src.utils import transform_data

# ==============================================================================
# 2. Análisis y manipulación de datos
# ==============================================================================
import numpy as np
import pandas as pd
import polars as pl

# ==============================================================================
# 3. Modelado y optimización (Machine Learning & Deep Learning)
# ==============================================================================
# Modelos tabulares y optimización bayesiana
from lightgbm import LGBMClassifier, LGBMRegressor
import optuna

# Redes neuronales (PyTorch)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Temporal Fusion Transformer (Modelos de secuencia)
try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import TFT
    from neuralforecast.losses.pytorch import HuberLoss
except ImportError:
    pass

# Ingeniería de variables automática (tsfresh)
from tsfresh import extract_features, select_features
from tsfresh.feature_extraction.settings import EfficientFCParameters, from_columns
from tsfresh.utilities.dataframe_functions import impute, roll_time_series

# ==============================================================================
# 4. Visualización
# ==============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import shap

# ==============================================================================
# 5. Módulos locales del proyecto (detour_analyzer)
# ==============================================================================
# Asegurar que la raíz del proyecto esté en el path (si ejecutas desde subcarpetas)
project_root = os.path.abspath(os.path.join(os.getcwd()))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Importaciones del módulo src
from src.features.build_features import TSFreshFeatureExtractor

# 1. Carga y transformación de los datos


In [2]:
df_raw = pl.read_parquet(r"G:\Mi unidad\Python\detour_analyzer\data\historico_consumo.parquet")
df_t = transform_data(df_raw).filter(pl.col("strategy") == "VMI", pl.col("fecha") >= pl.datetime(2026, 1, 1))
df_t = df_t.with_columns(
    id = (pl.col("planta") + "_"  + pl.col("sku")),
).filter(
    pl.col("id").is_in(["PCEL_HP2_01_115_1990"])
)

df_t

planta,sku,fecha,stock_actual,consumo_real,consumo_real_to,forecast_mensual,daily_forecast,target_is_peak,target_consumo_real,grade,subgroup,grammage,width,product_code,strategy,safety_days,safety_stock_to,hist_month_to_date_to,fcst_month_to_date_to,dev_hist_fcst_month_to_date_to,dev_hist_fcst_month_to_date_pct,stock_vs_safety_stock_to,stock_vs_safety_stock_pct,stock_en_pitea,pedido_nov,id
str,str,date,f64,f64,f64,f64,f64,i8,f64,str,str,i64,i64,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str
"""PCEL""","""HP2_01_115_1990""",2026-01-02,35.006,2.389,2.389,81.001,4.05005,0,3.883,"""HP2""","""01""",115,1990,942800,"""VMI""",9,34.715,0.0,10.800133,-10.800133,-100.0,0.291,0.838254,null,null,"""PCEL_HP2_01_115_1990"""
"""PCEL""","""HP2_01_115_1990""",2026-01-05,28.732,3.883,3.883,81.001,4.05005,0,2.391,"""HP2""","""01""",115,1990,942800,"""VMI""",9,36.644,3.883,16.2002,-12.3172,-76.03116,-7.912,-21.591529,null,null,"""PCEL_HP2_01_115_1990"""
"""PCEL""","""HP2_01_115_1990""",2026-01-07,24.545,2.391,2.391,81.001,4.05005,0,4.187,"""HP2""","""01""",115,1990,942800,"""VMI""",9,37.607,6.274,18.900233,-12.626233,-66.804643,-13.062,-34.732895,null,null,"""PCEL_HP2_01_115_1990"""
"""PCEL""","""HP2_01_115_1990""",2026-01-08,35.153,4.187,4.187,81.001,4.05005,0,4.866,"""HP2""","""01""",115,1990,942800,"""VMI""",9,38.572,10.461,21.600267,-11.139267,-51.570042,-3.419,-8.863943,null,null,"""PCEL_HP2_01_115_1990"""
"""PCEL""","""HP2_01_115_1990""",2026-01-09,47.106,4.866,4.866,81.001,4.05005,0,0.0,"""HP2""","""01""",115,1990,942800,"""VMI""",9,38.572,15.327,29.700367,-14.373367,-48.394576,8.534,22.124857,null,null,"""PCEL_HP2_01_115_1990"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""PCEL""","""HP2_01_115_1990""",2026-05-20,0.0,7.817,7.817,45.0,2.25,1,4.563,"""HP2""","""01""",115,1990,942800,"""VMI""",7,4.6,58.185,30.0,28.185,93.95,-4.6,-100.0,null,null,"""PCEL_HP2_01_115_1990"""
"""PCEL""","""HP2_01_115_1990""",2026-05-21,16.012,4.563,4.563,45.0,2.25,1,4.617,"""HP2""","""01""",115,1990,942800,"""VMI""",7,5.6,62.748,31.5,31.248,99.2,10.412,185.928571,null,null,"""PCEL_HP2_01_115_1990"""
"""PCEL""","""HP2_01_115_1990""",2026-05-22,13.656,4.617,4.617,45.0,2.25,0,0.0,"""HP2""","""01""",115,1990,942800,"""VMI""",7,19.6,67.365,36.0,31.365,87.125,-5.944,-30.326531,null,null,"""PCEL_HP2_01_115_1990"""


In [10]:
import pandas as pd
from src.features.feature_engine_features import FeatureEngineBuilder

# 1. Crear un DataFrame sintético de entrenamiento
data_train = {
    "planta": ["A", "A", "A", "A", "B", "B", "B", "B"],
    "sku": ["SKU1", "SKU1", "SKU1", "SKU1", "SKU2", "SKU2", "SKU2", "SKU2"],
    "fecha": pd.to_datetime([
        "2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04",
        "2026-01-01", "2026-01-02", "2026-01-03", "2026-01-04"
    ]),
    "consumo_real": [100.0, 150.0, 200.0, 250.0, 1_000.0, 1_100.0, 1_200.0, 1_300.0]
}
df_train = pd.DataFrame(data_train)

# 2. Inicializar el FeatureEngineBuilder
builder = FeatureEngineBuilder(
    value_cols="consumo_real",
    group_cols=("planta", "sku"),
    date_col="fecha",
    lags=(1, 2),
    windows=(2, 3),
    window_functions=("mean", "max"),
    add_expanding=True,
    expanding_functions=("mean",)
)

# 3. Ajustar el builder y transformar los datos de entrenamiento
df_train_feat, feature_cols = builder.fit_transform(df_train)

print("Columnas de características generadas:")
print(feature_cols)

print("\nDataFrame con las nuevas columnas:")
print(df_train_feat)

# 4. Transformar nuevos datos (por ejemplo, conjunto de test)
data_test = {
    "planta": ["A", "B"],
    "sku": ["SKU1", "SKU2"],
    "fecha": pd.to_datetime(["2026-01-05", "2026-01-05"]),
    "consumo_real": [300.0, 1_400.0]
}
df_test = pd.DataFrame(data_test)

# Se concatena con el histórico para poder calcular los retardos y ventanas móviles correspondientes
df_full_test = pd.concat([df_train, df_test]).reset_index(drop=True)
df_test_feat, _ = builder.transform(df_full_test)

# Filtrar para obtener solo las predicciones de la fecha de test
print("\nCaracterísticas calculadas para el conjunto de test (fecha 2026-01-05):")
display(df_test_feat[df_test_feat["fecha"] == "2026-01-05"])


Columnas de características generadas:
['consumo_real_lag_1', 'consumo_real_lag_2', 'consumo_real_window_2_mean', 'consumo_real_window_2_max', 'consumo_real_window_3_mean', 'consumo_real_window_3_max', 'consumo_real_expanding_mean']

DataFrame con las nuevas columnas:
  planta   sku      fecha  consumo_real  consumo_real_lag_1  \
0      A  SKU1 2026-01-01         100.0                 NaN   
1      A  SKU1 2026-01-02         150.0               100.0   
2      A  SKU1 2026-01-03         200.0               150.0   
3      A  SKU1 2026-01-04         250.0               200.0   
4      B  SKU2 2026-01-01        1000.0                 NaN   
5      B  SKU2 2026-01-02        1100.0              1000.0   
6      B  SKU2 2026-01-03        1200.0              1100.0   
7      B  SKU2 2026-01-04        1300.0              1200.0   

   consumo_real_lag_2  consumo_real_window_2_mean  consumo_real_window_2_max  \
0                 NaN                         NaN                        NaN   
1  

,planta,sku,fecha,consumo_real,consumo_real_lag_1,consumo_real_lag_2,consumo_real_window_2_mean,consumo_real_window_2_max,consumo_real_window_3_mean,consumo_real_window_3_max,consumo_real_expanding_mean
4,A,SKU1,2026-01-05,300.0,250.0,200.0,225.0,250.0,200.0,250.0,175.0
9,B,SKU2,2026-01-05,1400.0,1300.0,1200.0,1250.0,1300.0,1200.0,1300.0,1150.0
